# PhenoAgeSaoPaulo

## Index
1. [Instantiate model class](#Instantiate-model-class)
2. [Define clock metadata](#Define-clock-metadata)
3. [Download clock dependencies](#Download-clock-dependencies)
4. [Load features](#Load-features)
5. [Load weights into base model](#Load-weights-into-base-model)
6. [Load reference values](#Load-reference-values)
7. [Load preprocess and postprocess objects](#Load-preprocess-and-postprocess-objects)
8. [Check all clock parameters](#Check-all-clock-parameters)
9. [Normal feature ranges](#Normal-feature-ranges)
10. [Basic test](#Basic-test)
11. [Save torch model](#Save-torch-model)
12. [Clear directory](#Clear-directory)

Let's first import some packages:

In [1]:
import os
import inspect
import shutil
import subprocess
import json
import math
import torch
import pandas as pd
import pyaging as pya

## Instantiate model class

In [2]:
def print_entire_class(cls):
    source = inspect.getsource(cls)
    print(source)

print_entire_class(pya.models.PhenoAgeSaoPaulo)

class PhenoAgeSaoPaulo(pyagingModel):
    """PhenoAge refit without creatinine, albumin, and alkaline phosphatase."""

    def __init__(self):
        super().__init__()
        for name in ["m_n", "m_d", "ba_n", "ba_d", "ba_i"]:
            self.register_buffer(name, torch.empty(0))

    def preprocess(self, x):
        """Apply BioAge's log1p transform to C-reactive protein, not ``PhenoAge``'s ``ln``."""
        return log1p_crp(self.features, x)

    def postprocess(self, x):
        """Convert the Gompertz mortality score to phenotypic age.

        Notes
        -----
        The constants are refit alongside the coefficients and differ from
        Levine's published ones, so they travel as buffers rather than being
        hardcoded the way ``PhenoAge.postprocess`` hardcodes them.
        """
        m_n, m_d, ba_n, ba_d, ba_i = (
            buffer.to(device=x.device, dtype=x.dtype)
            for buffer in (self.m_n, self.m_d, self.ba_n, self.ba_d, self.ba_i)
        )
      

In [3]:
model = pya.models.PhenoAgeSaoPaulo()

## Define clock metadata

In [4]:
model.metadata["clock_name"] = 'phenoagesaopaulo'
model.metadata["data_type"] = 'clinical biomarkers'  # Paper: a novel measure of 'phenotypic age' was developed using clinical data from the third National Health and Nutrition Examination Survey (NHANES)
model.metadata["species"] = 'Homo sapiens'  # Paper: our analytical sample included 9,926 adults with complete biomarker data
model.metadata["year"] = 2026
model.metadata["approved_by_author"] = '⌛'
model.metadata["citation"] = [
    'Kwon, Dayoon, and Daniel W. Belsky. "A toolkit for quantification of biological age from blood chemistry and organ function test data: BioAge." GeroScience 43.6 (2021): 2795-2808.',
    'Levine, M. E., et al. "An epigenetic biomarker of aging for lifespan and healthspan." Aging 10.4 (2018): 573-591.',
]
model.metadata["doi"] = 'https://doi.org/10.1007/s11357-021-00480-5'
model.metadata["notes"] = "PhenoAge refit on NHANES III adults aged 20-84 with BioAge::phenoage_calc(), dropping creatinine, albumin, and alkaline phosphatase and keeping the remaining six biomarkers plus chronological age. The coefficients and the mortality-to-age constants are refit, so they differ from the published PhenoAge and its constants do not apply. Biomarkers are on BioAge's SI-unit variants, which are natively pyaging's unit convention, and C-reactive protein is supplied raw in mg/dL and log1p-transformed inside the clock. The refit is pooled across sexes, so unlike kdmage and homeostaticdysregulation this clock takes no female column. The refit has no publication of its own: it was produced for pyaging 0.5.0 by clocks/notebooks/phenoagesaopaulo.ipynb with the BioAge toolkit, so the DOI, journal, last author and citation count are the toolkit's, and Levine 2018 is listed beside it as the method reference that defines PhenoAge."
model.metadata["research_only"] = None
model.metadata["tissue"] = ['blood']  # Paper: measures like CRP, albumin, creatinine, glucose, etc.
model.metadata["predicts"] = ['phenotypic age']  # Paper: These nine biomarkers and chronological age were then combined in a phenotypic age estimate (in units of years)
model.metadata["training_target"] = ['mortality']  # Paper: gom = flexsurvreg(surv_form(bm_name), data = dat, dist = "gompertz")
model.metadata["unit"] = ['years']  # Paper: the mortality score was converted into units of years
model.metadata["model_type"] = 'Gompertz hazards regression with age calibration'  # Paper: These nine biomarkers and chronological age were then included in a parametric proportional hazards model based on the Gompertz distribution. Based on this model, we estimated the 10-year (120 months) mortality risk of the j-the individual. Next, the mortality score was converted into units of years
model.metadata["platform"] = ['clinical laboratory assays']  # Paper: forty-two clinical markers
model.metadata["population"] = 'adults'  # Paper: filter(age >= 20, age <= 84)
model.metadata["journal"] = 'GeroScience'  # Paper: GeroScience 43(6)
model.metadata["last_author"] = 'Daniel W. Belsky'  # Paper: author list
model.metadata["n_features"] = 7  # Code: six biomarkers plus age
model.metadata["citations"] = 332  # Paper: the BioAge toolkit paper, shared with kdmage/homeostaticdysregulation
model.metadata["citations_date"] = '2026-08-20'

## Download clock dependencies

In [5]:
# BioAge carries both the fitting functions and the NHANES III / NHANES IV tables they run
# on, so the parameters and the parity reference below are re-derived here rather than read
# from a checked-in copy. The script needs R on PATH; it installs what it is missing into a
# notebook-local library that the Clear directory step removes.
EXTRACT_R = r"""#!/usr/bin/env Rscript
# Refit PhenoAge with dayoonkwon/BioAge, which ships both phenoage_calc() and
# the NHANES III / NHANES IV tables it runs on, dropping creatinine, albumin and
# alkaline phosphatase and keeping the remaining six biomarkers plus age.
#
# The refit is trained on SI-unit variants of the NHANES columns, so the
# coefficients land natively in pyaging's unit convention: a Gompertz
# coefficient scales as 1/c under a unit change of c, so fitting in SI units
# reproduces BioAge's own output exactly rather than approximating it.
#
# Unit notes, established empirically against the shipped NHANES data:
#   * lncrp is log1p(crp in mg/dL), NOT log(crp): exp(lncrp) - crp == 1 exactly
#     across both cohorts.
#   * glucose_mmol == glucose * 0.0555 is exact, with zero deviation across
#     every non-missing row of both cohorts.
#
# CRP naming. The fitted column stays `log_crp`, because that is what the value
# is: log1p(CRP in mg/dL). The name EMITTED is `c_reactive_protein`, because
# that is what a pyaging user supplies -- the raw measurement in mg/dL, which
# the clock log1p's itself in preprocess(). The rename is name-only: no
# coefficient or training mean moves. The emitted reference rows carry raw
# `crp`, so feeding them in and letting the clock transform reproduces BioAge's
# own output.

local_library <- file.path(getwd(), "Rlib")
dir.create(local_library, showWarnings = FALSE)
.libPaths(c(local_library, .libPaths()))
for (package in c("remotes", "dplyr", "jsonlite")) {
  if (!requireNamespace(package, quietly = TRUE)) {
    install.packages(package, repos = "https://cloud.r-project.org", lib = local_library)
  }
}
if (!requireNamespace("BioAge", quietly = TRUE)) {
  remotes::install_github("dayoonkwon/BioAge@b1f9fc02f086cd4aa74185f2335ab1366082e7fe", lib = local_library, upgrade = "never")
}

suppressPackageStartupMessages({
  library(BioAge)
  library(dplyr)
  library(jsonlite)
})

to_pyaging_units <- function(data) {
  data %>% mutate(
    glucose = glucose_mmol,
    log_crp = lncrp,
    c_reactive_protein = crp,
    lymphocyte_percent = lymph,
    mean_cell_volume = mcv,
    red_cell_distribution_width = rdw,
    white_blood_cell_count = wbc,
    female = as.numeric(gender == 2)
  )
}

nhanes3 <- to_pyaging_units(NHANES3)
nhanes4 <- to_pyaging_units(NHANES4)

to_feature_names <- function(names) replace(names, names == "log_crp", "c_reactive_protein")

markers <- c(
  "glucose", "log_crp", "lymphocyte_percent",
  "mean_cell_volume", "red_cell_distribution_width", "white_blood_cell_count"
)

training <- nhanes3 %>% filter(age >= 20, age <= 84)
train <- phenoage_calc(training, biomarkers = markers, fit = NULL)

# flexsurvreg's gompertz coefficient table is rownamed
# c("shape", "rate", <covariates...>); "rate" is the linear-predictor intercept.
coefficients <- train$fit$coef
stopifnot(identical(rownames(coefficients), c("shape", "rate", markers, "age")))

# Training-sample means over exactly the rows the gompertz fit used: flexsurvreg
# drops incomplete cases, so the model frame, not the filtered data frame, is
# the training sample. pyaging substitutes these for any predictor a user's data
# does not carry, so an absent one contributes its average contribution to the
# linear predictor instead of the 0 the pipeline would otherwise supply. They
# are on the fitted scale, so the log_crp entry is log1p(CRP in mg/dL).
frame <- model.frame(
  BioAge:::surv_form(paste(c(markers, "age"), collapse = "+")),
  data = training
)
means <- colMeans(frame[, c(markers, "age")])

# ---- Reference predictions, for the parity check ---------------------------
# 20 fixed NHANES IV subjects: complete cases across every column this clock
# consumes, sorted by sampleID (C-locale byte order), first 20. `expected`
# comes from BioAge's own phenoage_calc, never from a re-implementation.
projected <- phenoage_calc(
  nhanes4 %>% filter(age >= 20),
  biomarkers = markers, fit = train$fit
)$data %>% select(sampleID, phenoage)

# Subject selection runs over the fitted columns, so carrying the raw CRP column
# cannot shift cohort membership and move `expected`. It is joined back after.
selected <- nhanes4 %>%
  select(sampleID, all_of(c(markers, "age"))) %>%
  filter(stats::complete.cases(.)) %>%
  arrange(sampleID) %>%
  head(20) %>%
  left_join(nhanes4 %>% select(sampleID, c_reactive_protein), by = "sampleID") %>%
  left_join(projected, by = "sampleID")

stopifnot(nrow(selected) == 20, !anyNA(selected))

# The emitted rows carry raw CRP where the fit carries log1p(CRP); the clock
# closes that gap in preprocess(). If this fails, the two have drifted apart.
stopifnot(max(abs(log1p(selected$c_reactive_protein) - selected$log_crp)) < 1e-12)

emit_features <- to_feature_names(c(markers, "age"))

write_json(
  list(
    features = emit_features,
    coefficients = as.numeric(coefficients[c(markers, "age"), "coef"]),
    intercept = as.numeric(coefficients["rate", "coef"]),
    training_mean = as.numeric(means[c(markers, "age")]),
    training_n = nrow(frame),
    m_n = as.numeric(train$fit$m_n),
    m_d = as.numeric(train$fit$m_d),
    ba_n = as.numeric(train$fit$BA_n),
    ba_d = as.numeric(train$fit$BA_d),
    ba_i = as.numeric(train$fit$BA_i),
    reference = list(
      sample_ids = selected$sampleID,
      rows = selected %>% select(all_of(emit_features)),
      expected = selected$phenoage
    )
  ),
  "phenoagesaopaulo.json",
  digits = 12, auto_unbox = TRUE, pretty = TRUE
)

cat("wrote phenoagesaopaulo.json\n")
"""

with open("extract_phenoagesaopaulo.R", "w") as handle:
    handle.write(EXTRACT_R)

subprocess.run(["Rscript", "extract_phenoagesaopaulo.R"], check=True)

wrote phenoagesaopaulo.json


CompletedProcess(args=['Rscript', 'extract_phenoagesaopaulo.R'], returncode=0)

In [6]:
with open("phenoagesaopaulo.json") as handle:
    params = json.load(handle)

params["features"]

['glucose',
 'c_reactive_protein',
 'lymphocyte_percent',
 'mean_cell_volume',
 'red_cell_distribution_width',
 'white_blood_cell_count',
 'age']

## Load features

In [7]:
model.features = params["features"]
model.features

['glucose',
 'c_reactive_protein',
 'lymphocyte_percent',
 'mean_cell_volume',
 'red_cell_distribution_width',
 'white_blood_cell_count',
 'age']

## Load weights into base model

In [8]:
base_model = pya.models.LinearModel(input_dim=len(model.features))
base_model.linear.weight.data = torch.tensor([params["coefficients"]], dtype=torch.float64)
base_model.linear.bias.data = torch.tensor([params["intercept"]], dtype=torch.float64)
model.base_model = base_model

for name in ["m_n", "m_d", "ba_n", "ba_d", "ba_i"]:
    setattr(model, name, torch.tensor(params[name], dtype=torch.float64))

pd.DataFrame(
    {"constant": ["m_n", "m_d", "ba_n", "ba_d", "ba_i"],
     "refit": [params[name] for name in ["m_n", "m_d", "ba_n", "ba_d", "ba_i"]],
     "levine_published": [-1.51714, 0.007692696, -0.0055305, 0.090165, 141.50225]}
)

,constant,refit,levine_published
0,m_n,-1.357302,-1.517140
1,m_d,0.007146,0.007693
2,ba_n,-0.005808,-0.005530
3,ba_d,0.087422,0.090165
4,ba_i,141.825450,141.502250


## Load reference values

In [9]:
crp = model.features.index("c_reactive_protein")
reference = list(params["training_mean"])
reference[crp] = math.expm1(reference[crp])  # stored raw; preprocess applies log1p

model.reference_values = reference

assert len(model.reference_values) == len(model.features)
pd.DataFrame({"feature": model.features, "reference_value": model.reference_values})

,feature,reference_value
0,glucose,5.447648
1,c_reactive_protein,0.376398
2,lymphocyte_percent,33.035487
3,mean_cell_volume,89.421693
4,red_cell_distribution_width,13.149559
5,white_blood_cell_count,7.171985
6,age,47.073023


## Load preprocess and postprocess objects

In [10]:
model.preprocess_name = "log1p_crp"
model.preprocess_dependencies = None

In [11]:
model.postprocess_name = "mortality_to_phenoage_saopaulo"
model.postprocess_dependencies = None

## Check all clock parameters

In [12]:
pya.utils.print_model_details(model)


%==================================== Model Details ====================================%
Model Attributes:

training: True
metadata: {'approved_by_author': '⌛',
 'citation': ['Kwon, Dayoon, and Daniel W. Belsky. "A toolkit for '
              'quantification of biological age from blood chemistry and organ '
              'function test data: BioAge." GeroScience 43.6 (2021): '
              '2795-2808.',
              'Levine, M. E., et al. "An epigenetic biomarker of aging for '
              'lifespan and healthspan." Aging 10.4 (2018): 573-591.'],
 'citations': 332,
 'citations_date': '2026-08-20',
 'clock_name': 'phenoagesaopaulo',
 'data_type': 'clinical biomarkers',
 'doi': 'https://doi.org/10.1007/s11357-021-00480-5',
 'journal': 'GeroScience',
 'last_author': 'Daniel W. Belsky',
 'model_type': 'Gompertz hazards regression with age calibration',
 'n_features': 7,
 'notes': 'PhenoAge refit on NHANES III adults aged 20-84 with '
          'BioAge::phenoage_calc(), dropping crea

## Normal feature ranges

In [13]:
feature_ranges = pya.utils.resolve_feature_ranges(model.features, model.metadata["data_type"])
model.feature_units = [record["unit"] for record in feature_ranges]
pd.DataFrame.from_records(feature_ranges)

,feature,unit,low,high
0,glucose,mmol/L,1.00,60.0
1,c_reactive_protein,mg/dL,0.01,50.0
2,lymphocyte_percent,%,0.00,100.0
3,mean_cell_volume,fL,40.00,150.0
4,red_cell_distribution_width,%,8.00,40.0
5,white_blood_cell_count,10^3 cells/uL,0.05,500.0
6,age,years,0.00,122.5


## Basic test

In [14]:
records = pya.utils.resolve_feature_ranges(model.features, model.metadata["data_type"])
midpoints = torch.tensor(
    [[(record["low"] + record["high"]) / 2 for record in records]], dtype=torch.float64
)
model.eval()
model.to(torch.float64)
pred = model(midpoints)
pred

tensor([[inf]], dtype=torch.float64, grad_fn=<AddBackward0>)

#### Parity with BioAge

In [15]:
reference_predictions = params["reference"]
matrix = torch.tensor(
    [[row[name] for name in model.features] for row in reference_predictions["rows"]], dtype=torch.float64
)
with torch.inference_mode():
    predicted = model(matrix).squeeze(-1)

expected = torch.tensor(reference_predictions["expected"], dtype=torch.float64)
print("max absolute difference:", (predicted - expected).abs().max().item())

max absolute difference: 3.0411229090532288e-12


## Save torch model

In [16]:
torch.save(model, f"../weights/{model.metadata['clock_name']}.pt")

## Clear directory
<a id="10"></a>

In [17]:
# Function to remove a folder and all its contents
def remove_folder(path):
    try:
        shutil.rmtree(path)
        print(f"Deleted folder: {path}")
    except Exception as e:
        print(f"Error deleting folder {path}: {e}")

# Get a list of all files and folders in the current directory
all_items = os.listdir('.')

# Loop through the items
for item in all_items:
    # Check if it's a file and does not end with .ipynb
    if os.path.isfile(item) and not item.endswith('.ipynb'):
        os.remove(item)
        print(f"Deleted file: {item}")
    # Check if it's a folder
    elif os.path.isdir(item):
        remove_folder(item)

Deleted file: extract_phenoagesaopaulo.R
Deleted folder: Rlib
Deleted file: phenoagesaopaulo.json
